聚类代码

声子频率标准化代码：由于材料的声子频率长度不一，所以要对其做标准化处理，具体处理方式为找到能够包含90%材料的声子频率长度，以此为基准，高于该长度截断，低于该长度补0。

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import ast
from tqdm import tqdm

# Configuration
DATASET_PATH = './dataset/dataset.csv'
OUTPUT_DIR = './standardized_frequencies'
OUTPUT_FILENAME = 'materials_standardized_frequencies.csv'


def safe_literal_eval(val):
    """
    Safely convert string representation of list or tuple to actual list.
    Returns empty list if conversion fails.
    """
    if isinstance(val, str):
        try:
            evaluated_obj = ast.literal_eval(val)
            if isinstance(evaluated_obj, (list, tuple)):
                return list(evaluated_obj)
            return []
        except (ValueError, SyntaxError):
            return []
    return []


def standardize_frequency_list(freq_list, target_length):
    """
    Standardize a single frequency list to target length.
    - If list is too long, truncate it.
    - If list is too short, pad with zeros at the end.
    """
    current_length = len(freq_list)
    if current_length > target_length:
        return freq_list[:target_length]
    elif current_length < target_length:
        padding = [0.0] * (target_length - current_length)
        return freq_list + padding
    return freq_list


def main():
    """Main execution pipeline"""
    print("=== Phonon Frequency Standardization Pipeline ===")
    
    # 1. Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Output directory confirmed: {OUTPUT_DIR}")

    # 2. Load dataset
    print(f"\nStep 1/6: Loading dataset: {DATASET_PATH}")
    try:
        data = pd.read_csv(DATASET_PATH)
        print(f"  Successfully loaded {len(data)} material data points.")
        initial_ids = set(data['id'])
    except FileNotFoundError:
        print(f"  Error: Dataset file not found! Please check the path.")
        return
    
    # 3. Parse frequency data and calculate lengths
    print("\nStep 2/6: Parsing phonon frequencies and calculating lengths...")
    # Filter out rows where 'frequency' column is empty
    data.dropna(subset=['frequency'], inplace=True)
    
    tqdm.pandas(desc="  Parsing frequency strings")
    data['frequency_list'] = data['frequency'].progress_apply(safe_literal_eval)
    data['frequency_length'] = data['frequency_list'].apply(len)
    
    # Filter out invalid data with zero length after parsing
    initial_count_with_freq_col = len(data)
    data = data[data['frequency_length'] > 0].copy()
    valid_count = len(data)
    print(f"  Parsing completed. Valid phonon frequency data: {valid_count} entries "
          f"(filtered from {initial_count_with_freq_col} records with frequency column).")

    # 4. Check and report missing frequency materials
    print("\nStep 3/6: Checking frequency data integrity...")
    valid_ids = set(data['id'])
    missing_ids = initial_ids - valid_ids
    
    if not missing_ids:
        print("  ✅ Integrity check passed: All materials have valid phonon frequency data.")
    else:
        print(f"  ⚠️  Warning: Found {len(missing_ids)} materials missing valid phonon frequency data.")
        missing_ids_path = os.path.join(OUTPUT_DIR, 'materials_missing_frequencies.txt')
        with open(missing_ids_path, 'w') as f:
            for item_id in sorted(list(missing_ids)):
                f.write(f"{item_id}\n")
        print(f"  List of missing material IDs saved to: {missing_ids_path}")

    # 5. Determine standard length and visualize
    print("\nStep 4/6: Determining standard length (covering 90% of materials)...")
    # Calculate 90th percentile as standard length
    standard_length = int(data['frequency_length'].quantile(0.90))
    print(f"  Calculated standard length: {standard_length}")

    # Plot length distribution histogram
    plt.figure(figsize=(12, 7))
    plt.hist(data['frequency_length'], bins=100, color='skyblue', edgecolor='black', 
             alpha=0.8, range=(0, data['frequency_length'].quantile(0.99)))
    plt.axvline(standard_length, color='red', linestyle='--', linewidth=2, 
                label=f'90% Quantile Length = {standard_length}')
    plt.title('Distribution of Phonon Frequency Lengths', fontsize=16)
    plt.xlabel('Number of Frequencies per Material', fontsize=12)
    plt.ylabel('Number of Materials', fontsize=12)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    histogram_path = os.path.join(OUTPUT_DIR, 'frequency_length_distribution.png')
    plt.savefig(histogram_path, dpi=300)
    plt.close()
    print(f"  Length distribution histogram saved to: {histogram_path}")

    # 6. Standardize frequency data
    print("\nStep 5/6: Standardizing all frequency data to standard length...")
    tqdm.pandas(desc="  Standardization processing")
    data['standardized_frequency'] = data['frequency_list'].progress_apply(
        lambda x: standardize_frequency_list(x, standard_length)
    )

    # 7. Save results
    print("\nStep 6/6: Saving standardized dataset...")
    final_df = data[['id', 'standardized_frequency']]
    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
    final_df.to_csv(output_path, index=False)
    
    print("="*60)
    print("✅ Phonon frequency standardization pipeline completed!")
    print(f"  - Total {valid_count} materials processed.")
    print(f"  - All material phonon frequencies standardized to length {standard_length}.")
    print(f"  - Results saved to: {output_path}")
    print("="*60)


if __name__ == "__main__":
    main()


=== Phonon Frequency Standardization Pipeline ===
Output directory confirmed: ./standardized_frequencies

Step 1/6: Loading dataset: ./dataset/dataset.csv
  Successfully loaded 11648 material data points.

Step 2/6: Parsing phonon frequencies and calculating lengths...


  Parsing frequency strings: 100%|██████████| 11648/11648 [00:00<00:00, 23932.19it/s]


  Parsing completed. Valid phonon frequency data: 11648 entries (filtered from 11648 records with frequency column).

Step 3/6: Checking frequency data integrity...
  ✅ Integrity check passed: All materials have valid phonon frequency data.

Step 4/6: Determining standard length (covering 90% of materials)...
  Calculated standard length: 60
  Length distribution histogram saved to: ./standardized_frequencies/frequency_length_distribution.png

Step 5/6: Standardizing all frequency data to standard length...


  Standardization processing: 100%|██████████| 11648/11648 [00:00<00:00, 107518.56it/s]


Step 6/6: Saving standardized dataset...


✅ Phonon frequency standardization pipeline completed!
  - Total 11648 materials processed.
  - All material phonon frequencies standardized to length 60.
  - Results saved to: ./standardized_frequencies/materials_standardized_frequencies.csv


为了找到对训练最有力的声子频率长度，对声子频率进行批次截断，即从6到上一步得到的标准长度，每隔一个长度进行截断。例子中使用的是6到78，每隔6截断一次。

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import ast
from tqdm import tqdm

# Configuration
DATASET_PATH = './dataset/dataset.csv'
OUTPUT_DIR = './frequency_cuts'
OUTPUT_FILENAME = 'materials_standardized_frequencies.csv'


def safe_literal_eval(val):
    """
    Safely convert string representation of list or tuple to actual list.
    Returns empty list if conversion fails.
    """
    if isinstance(val, str):
        try:
            evaluated_obj = ast.literal_eval(val)
            if isinstance(evaluated_obj, (list, tuple)):
                return list(evaluated_obj)
            return []
        except (ValueError, SyntaxError):
            return []
    return []


def standardize_frequency_list(freq_list, target_length):
    """
    Standardize a single frequency list to target length.
    - If list is too long, truncate it.
    - If list is too short, pad with zeros at the end.
    """
    current_length = len(freq_list)
    if current_length > target_length:
        return freq_list[:target_length]
    elif current_length < target_length:
        padding = [0.0] * (target_length - current_length)
        return freq_list + padding
    return freq_list


def main():
    """Main execution pipeline"""
    print("=== Batch Phonon Frequency Standardization Pipeline ===")
    
    # 1. Create root output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Root output directory confirmed: {OUTPUT_DIR}")

    # 2. Load dataset
    print(f"\nStep 1/4: Loading dataset: {DATASET_PATH}")
    try:
        data = pd.read_csv(DATASET_PATH)
        print(f"  Successfully loaded {len(data)} material data points.")
        initial_ids = set(data['id'])
    except FileNotFoundError:
        print(f"  Error: Dataset file not found! Please check the path.")
        return
    
    # 3. Parse frequency data and calculate lengths
    print("\nStep 2/4: Parsing phonon frequencies...")
    # Filter out rows where 'frequency' column is empty
    data.dropna(subset=['frequency'], inplace=True)
    
    tqdm.pandas(desc="  Parsing frequency strings")
    data['frequency_list'] = data['frequency'].progress_apply(safe_literal_eval)
    data['frequency_length'] = data['frequency_list'].apply(len)
    
    # Filter out invalid data with zero length after parsing
    initial_count_with_freq_col = len(data)
    data = data[data['frequency_length'] > 0].copy()
    valid_count = len(data)
    print(f"  Parsing completed. Valid phonon frequency data: {valid_count} entries "
          f"(filtered from {initial_count_with_freq_col} records with frequency column).")

    # 4. Check and report missing frequency materials
    print("\nStep 3/4: Checking frequency data integrity...")
    valid_ids = set(data['id'])
    missing_ids = initial_ids - valid_ids
    
    if not missing_ids:
        print("  ✅ Integrity check passed: All materials have valid phonon frequency data.")
    else:
        print(f"  ⚠️  Warning: Found {len(missing_ids)} materials missing valid phonon frequency data.")
        missing_ids_path = os.path.join(OUTPUT_DIR, 'materials_missing_frequencies.txt')
        with open(missing_ids_path, 'w') as f:
            for item_id in sorted(list(missing_ids)):
                f.write(f"{item_id}\n")
        print(f"  List of missing material IDs saved to: {missing_ids_path}")

    # 5. Process each target length in a loop
    print("\nStep 4/4: Processing each truncation length (from 6 to 78, step 6)...")
    for target_length in range(6, 61, 6):
        print(f"\n--- Processing length: {target_length} ---")
        
        # Create output subdirectory for current length
        current_output_dir = os.path.join(OUTPUT_DIR, str(target_length))
        os.makedirs(current_output_dir, exist_ok=True)
        
        # Standardize frequency data
        tqdm.pandas(desc=f"  Standardizing (L={target_length})")
        data['standardized_frequency'] = data['frequency_list'].progress_apply(
            lambda x: standardize_frequency_list(x, target_length)
        )

        # Save results
        final_df = data[['id', 'standardized_frequency']]
        output_path = os.path.join(current_output_dir, OUTPUT_FILENAME)
        final_df.to_csv(output_path, index=False)
        print(f"  Results saved to: {output_path}")

    print("\n" + "="*60)
    print("✅ Batch phonon frequency standardization pipeline completed!")
    print(f"  - Total {valid_count} materials processed.")
    print(f"  - Results saved to respective length subfolders under: {OUTPUT_DIR}")
    print("="*60)


if __name__ == "__main__":
    main()


=== Batch Phonon Frequency Standardization Pipeline ===
Root output directory confirmed: ./frequency_cuts

Step 1/4: Loading dataset: ./dataset/dataset.csv
  Successfully loaded 11648 material data points.

Step 2/4: Parsing phonon frequencies...


  Parsing frequency strings: 100%|██████████| 11648/11648 [00:00<00:00, 22488.18it/s]


  Parsing completed. Valid phonon frequency data: 11648 entries (filtered from 11648 records with frequency column).

Step 3/4: Checking frequency data integrity...
  ✅ Integrity check passed: All materials have valid phonon frequency data.

Step 4/4: Processing each truncation length (from 6 to 78, step 6)...

--- Processing length: 6 ---


  Standardizing (L=6): 100%|██████████| 11648/11648 [00:00<00:00, 600859.11it/s]


  Results saved to: ./frequency_cuts/6/materials_standardized_frequencies.csv

--- Processing length: 12 ---


...

  Results saved to: ./frequency_cuts/48/materials_standardized_frequencies.csv

--- Processing length: 54 ---


  Standardizing (L=54): 100%|██████████| 11648/11648 [00:00<00:00, 406513.95it/s]


  Results saved to: ./frequency_cuts/54/materials_standardized_frequencies.csv

--- Processing length: 60 ---


  Standardizing (L=60): 100%|██████████| 11648/11648 [00:00<00:00, 362097.29it/s]


  Results saved to: ./frequency_cuts/60/materials_standardized_frequencies.csv

✅ Batch phonon frequency standardization pipeline completed!
  - Total 11648 materials processed.
  - Results saved to respective length subfolders under: ./frequency_cuts


和上面聚类部分一致，但是保存了kmeans模型和降维模型

In [2]:
import pandas as pd
import numpy as np
import matplotlib
# 设置后端为 Agg，适用于没有图形界面的远程服务器
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MaxAbsScaler    
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.impute import SimpleImputer
import umap
import os
import re
import warnings
import joblib  # [New] 引入 joblib 用于保存模型

# ==========================================
# 1. Configuration & Global Settings
# ==========================================
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.max_open_warning'] = 0

# --- 用户配置路径 ---
DATASET_PATH = '/home2/yhchen/01-PARCE/cluster_and_model/raman2/dataset/dataset.csv'
BASE_OUTPUT_DIR = "/home2/yhchen/01-PARCE/cluster_and_model/raman2/clustering_results_umap"

# --- 指定聚类任务 (Specific Tasks) ---
CLUSTERING_TASKS = [
    {'combination': '125', 'n_clusters': 50},   
    {'combination': '235', 'n_clusters': 50},
    {'combination': '135', 'n_clusters': 55},
    {'combination': '12345', 'n_clusters': 45},
]

# ==========================================
# 2. Helper Functions (New Logic)
# ==========================================

def _beta_to_sin_cos(beta_deg: pd.Series) -> pd.DataFrame:
    """把 β 角（度）转成 sin/cos 两维"""
    beta = pd.to_numeric(beta_deg, errors='coerce')
    beta_rad = np.deg2rad(beta)
    return pd.DataFrame({
        'beta_sin': np.sin(beta_rad),
        'beta_cos': np.cos(beta_rad),
    })

def _parse_wyckoff_string(s):
    """自定义 Wyckoff 解析器"""
    if pd.isna(s):
        return {}
    s = str(s).strip().lower()
    matches = re.findall(r'([a-z])(\d*)', s)
    counts = {}
    for char, count_str in matches:
        count = int(count_str) if count_str else 1
        counts[char] = counts.get(char, 0) + count
    return counts

def _parse_pearson_symbol(s):
    """自定义 Pearson 解析器"""
    s = str(s).strip()
    if len(s) < 3:
        return "Unknown", 0.0
    bravais = s[:2]
    try:
        atoms = float(s[2:])
    except:
        atoms = 0.0
    return bravais, atoms

# ==========================================
# 3. Improved Feature Engineering
# ==========================================

def process_all_features(data):
    """
    特征工程：
    拆解了链式调用，以便将所有的 scaler 和 imputer 对象独立保存。
    """
    print("  Starting feature engineering (Final Optimized Version)...")
    
    data = data.copy()
    data['space_group_number'] = pd.to_numeric(data['space_group_number'], errors='coerce').fillna(-1)

    # 1. Space Group (One-Hot)
    space_group_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', min_frequency=10)
    space_group_encoded = space_group_encoder.fit_transform(data[['space_group_number']].astype(int).astype(str))
    
    # 2. Pearson Symbol (Split Logic)
    pearson_parsed = data['pearson_symbol'].apply(_parse_pearson_symbol).tolist()
    bravais_list = [x[0] for x in pearson_parsed]
    atoms_list = [[x[1]] for x in pearson_parsed]

    # 2.1 Bravais (OneHot)
    bravais_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    bravais_encoded = bravais_encoder.fit_transform(np.array(bravais_list).reshape(-1, 1))
    
    # 2.2 Atoms (StandardScaler)
    atoms_scaler = StandardScaler()
    atoms_scaled = atoms_scaler.fit_transform(atoms_list)
    
    # Merge Pearson
    pearson_final = np.hstack([bravais_encoded, atoms_scaled])
    
    # 3. Wyckoff Sequence (Regex + DictVec + Scaler)
    wyckoff_dicts = data['wyckoff_sequence'].apply(_parse_wyckoff_string).tolist()
    vec = DictVectorizer(sparse=False, sort=True)
    wyckoff_raw_matrix = vec.fit_transform(wyckoff_dicts)
    
    # [Modified] 改为按最大值缩放，保持非负且 0 仍为 0
    wyckoff_scaler = MaxAbsScaler()
    wyckoff_final = wyckoff_scaler.fit_transform(wyckoff_raw_matrix)
    
    # 4. c/a Ratio
    ca = pd.to_numeric(data['c_a_ratio'], errors='coerce')
    # [Modified] 将 Imputer 和 Scaler 独立出来以便保存
    ca_imputer = SimpleImputer(strategy='median')
    ca_imputed = ca_imputer.fit_transform(ca.to_frame())
    ca_scaler = StandardScaler()
    ca_scaled = ca_scaler.fit_transform(ca_imputed)

    # 5. Beta Angle
    beta_sc = _beta_to_sin_cos(data['beta_angle'])
    # [Modified] 将 Imputer 和 Scaler 独立出来以便保存
    beta_imputer = SimpleImputer(strategy='median')
    beta_imputed = beta_imputer.fit_transform(beta_sc)
    beta_scaler = StandardScaler()
    beta_scaled = beta_scaler.fit_transform(beta_imputed)
    
    print("  Feature engineering completed.")

    # ==========================================
    # [NEW] 保存所有预处理器
    # ==========================================
    preprocessors = {
        'space_group_encoder': space_group_encoder,
        'bravais_encoder': bravais_encoder,
        'atoms_scaler': atoms_scaler,
        'wyckoff_vec': vec,
        'wyckoff_scaler': wyckoff_scaler,
        'ca_imputer': ca_imputer,
        'ca_scaler': ca_scaler,
        'beta_imputer': beta_imputer,
        'beta_scaler': beta_scaler
    }
    os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
    joblib.dump(preprocessors, os.path.join(BASE_OUTPUT_DIR, 'preprocessors.pkl'))
    print(f"  [Save] Preprocessors securely saved to: {os.path.join(BASE_OUTPUT_DIR, 'preprocessors.pkl')}")

    return {
        '1': space_group_encoded, 
        '2': pearson_final, 
        '3': wyckoff_final,
        '4': ca_scaled,
        '5': beta_scaled
    }

def get_selected_features(features_dict, selected_params):
    PHYSICAL_WEIGHTS = {
        '1': 1.0, '2': 1.0, '3': 1.0, '4': 3.0, '5': 3.0
    }
    feature_list = []
    for param_char in selected_params:
        if param_char in features_dict:
            raw = features_dict[param_char]
            w = PHYSICAL_WEIGHTS.get(param_char, 1.0)
            feature_list.append(raw * w)
    if not feature_list:
        raise ValueError(f"Feature combination '{selected_params}' is invalid.")
    return np.hstack(feature_list)


def run_specific_kmeans(data, features_dict, combination, n_clusters):
    print(f"\n{'='*60}")
    print(f"▶️  Processing: Combo={combination}, Clusters={n_clusters}")
    print(f"{'='*60}")
    
    output_dir = os.path.join(BASE_OUTPUT_DIR, combination, 'kmeans')
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"  Combining features: {combination}...")
    try:
        combined_features = get_selected_features(features_dict, combination)
        print(f"  Weighted feature matrix dimensions: {combined_features.shape}")
    except ValueError as e:
        print(f"  Error: {e}"); return

    # 3. UMAP Dimensionality Reduction
    print("  [Calculation] Running Global UMAP (10 dims) for K-means...")
    reducer_cluster = None
    if combined_features.shape[1] > 10:
        reducer_cluster = umap.UMAP(
            n_components=10, 
            n_neighbors=50,     
            min_dist=0.0,       
            metric='euclidean',
            random_state=42, 
            n_jobs=1
        )
        embedding_for_kmeans = reducer_cluster.fit_transform(combined_features)
        print(f"  UMAP Reduced dimensions: {combined_features.shape[1]} -> {embedding_for_kmeans.shape[1]}")
    else:
        embedding_for_kmeans = combined_features
        print("  Low dimensionality, skipping reduction.")

    # 4. Execute K-means clustering
    print(f"  Executing K-means clustering (K={n_clusters})...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embedding_for_kmeans)
    
    # ==========================================
    # [NEW] 保存模型：KMeans 与 UMAP (如果有)
    # ==========================================
    joblib.dump(kmeans, os.path.join(output_dir, f'kmeans_model_{combination}.pkl'))
    if reducer_cluster is not None:
        joblib.dump(reducer_cluster, os.path.join(output_dir, f'umap_reducer_{combination}.pkl'))
    print(f"  [Save] Machine learning models (.pkl) for combination {combination} successfully saved.")

    # 5. Evaluate clustering results
    silhouette_avg = silhouette_score(embedding_for_kmeans, cluster_labels)
    sse = kmeans.inertia_
    print(f"  Clustering evaluation completed:")
    print(f"    Silhouette Score: {silhouette_avg:.4f}")
    print(f"    SSE (Inertia): {sse:.2f}")

    # 6. Generate UMAP Coordinates (For Visualization ONLY)
    print("  [Visualization] Running UMAP (2 dims) to generate 2D coordinates...")
    reducer_viz = umap.UMAP(
        n_components=2, 
        n_neighbors=50,       
        min_dist=0.1,         
        metric='euclidean', 
        random_state=42, 
        n_jobs=1
    )
    umap_2d_embedding = reducer_viz.fit_transform(combined_features)

    # 7. Save Output Files
    print("  Saving output files...")
    clustered_data = data.copy()
    clustered_data['cluster'] = cluster_labels
    
    print("  Splitting data by cluster (5% test, 95% train)...")
    test_data_list = []
    train_data_list = []
    
    for cluster_id in range(n_clusters):
        current_cluster_subset = clustered_data[clustered_data['cluster'] == cluster_id]
        cluster_size = len(current_cluster_subset)
        
        if cluster_size > 0:
            test_size = max(1, int(np.ceil(cluster_size * 0.05)))
            test_subset = current_cluster_subset.sample(n=test_size, random_state=42)
            train_subset = current_cluster_subset.drop(test_subset.index)
            
            test_data_list.append(test_subset)
            train_data_list.append(train_subset)
    
    if test_data_list:
        test_data_final = pd.concat(test_data_list, ignore_index=True)
        test_data_final.to_csv(os.path.join(output_dir, 'test_pre.csv'), index=False)
        print(f"    Test data ({len(test_data_final)}) saved to test_pre.csv")
    
    if train_data_list:
        train_data_final = pd.concat(train_data_list, ignore_index=True)
        train_data_final.to_csv(os.path.join(output_dir, 'clustered_results.csv'), index=False)
        print(f"    Train data ({len(train_data_final)}) saved to clustered_results.csv")
    
    material_ids = data['id'] if 'id' in data.columns else data.index
    detailed_clustered_data = pd.DataFrame({
        'material_id': material_ids,
        'cluster': cluster_labels,
        'umap_dim_1': umap_2d_embedding[:, 0],
        'umap_dim_2': umap_2d_embedding[:, 1]
    })
    detailed_clustered_data.to_csv(os.path.join(output_dir, 'clustering_coordinates.csv'), index=False)
    
    cluster_stats = []
    for cluster_id in range(n_clusters):
        count = np.sum(cluster_labels == cluster_id)
        cluster_stats.append({
            'cluster_id': cluster_id,
            'material_count': count,
            'percentage': count / len(data) * 100
        })
    stats_df = pd.DataFrame(cluster_stats)
    stats_df.to_csv(os.path.join(output_dir, 'cluster_statistics.csv'), index=False)
    
    print(f"  All result files successfully saved in: {output_dir}")
    print(f"✅ Feature combination {combination} processing completed.")


def main():
    print("=== Specific Combination K-means Clustering (Final Optimized Version) ===")
    
    print(f"\nStep 1/3: Loading dataset: {DATASET_PATH}")
    try:
        data = pd.read_csv(DATASET_PATH)
        print(f"  Successfully loaded {len(data)} material data points.")
    except FileNotFoundError:
        print(f"  Error: Dataset file not found! Please check the path.")
        return

    print("\nStep 2/3: Preprocessing all features (Regex Wyckoff + Split Pearson)...")
    features_dict = process_all_features(data)

    print("\nStep 3/3: Processing specified clustering tasks...")
    for task in CLUSTERING_TASKS:
        run_specific_kmeans(
            data=data,
            features_dict=features_dict,
            combination=task['combination'],
            n_clusters=task['n_clusters']
        )
    
    print("\n" + "="*60)
    print("✅ All specified K-means clustering tasks completed!")
    print(f"📁 Results saved in root directory: {BASE_OUTPUT_DIR}")
    print("="*60)

if __name__ == "__main__":
    main()

=== Specific Combination K-means Clustering (Final Optimized Version) ===

Step 1/3: Loading dataset: /home2/yhchen/01-PARCE/cluster_and_model/raman2/dataset/dataset.csv
  Successfully loaded 11648 material data points.

Step 2/3: Preprocessing all features (Regex Wyckoff + Split Pearson)...
  Starting feature engineering (Final Optimized Version)...
  Feature engineering completed.
  [Save] Preprocessors securely saved to: /home2/yhchen/01-PARCE/cluster_and_model/raman2/clustering_results_umap/preprocessors.pkl

Step 3/3: Processing specified clustering tasks...

▶️  Processing: Combo=125, Clusters=50
  Combining features: 125...
  Weighted feature matrix dimensions: (11648, 111)
  [Calculation] Running Global UMAP (10 dims) for K-means...
  UMAP Reduced dimensions: 111 -> 10
  Executing K-means clustering (K=50)...
  [Save] Machine learning models (.pkl) for combination 125 successfully saved.
  Clustering evaluation completed:
    Silhouette Score: 0.8177
    SSE (Inertia): 14463.11

簇内声子频率平均，即将上一步Kmeans聚类的簇看作一个整体，计算簇内材料的平均声子频率来代表这个簇的声子频率，簇内平均声子频率也是下一步亲和传播聚类的输入参数。

In [3]:
import pandas as pd
import numpy as np
import os
import ast
from tqdm import tqdm

# Configuration
CLUSTER_BASE_DIR = "./clustering_results_umap/"
FREQ_BASE_DIR = "./frequency_cuts"

# Define clustering tasks to process (feature combinations and corresponding cluster counts)
CLUSTERING_TASKS = [
    {'combination': '125', 'n_clusters': 50},   
    {'combination': '235', 'n_clusters': 50},
    {'combination': '135', 'n_clusters': 55},
    {'combination': '12345', 'n_clusters': 45},
]

# Define frequency cutoff lengths to process
FREQUENCY_CUTOFFS = range(6, 61, 6)  # From 6 to 78, step 6


def safe_literal_eval(val):
    """Safely convert string representation of list or tuple to actual list."""
    if isinstance(val, str):
        try:
            evaluated_obj = ast.literal_eval(val)
            if isinstance(evaluated_obj, (list, tuple)):
                return list(evaluated_obj)
            return []
        except (ValueError, SyntaxError):
            return []
    return []


def main():
    """Main execution pipeline to calculate average phonon frequencies for each cluster"""
    print("=== Starting Average Phonon Frequency Calculation ===")

    # Iterate through each clustering task
    for task in CLUSTERING_TASKS:
        combination = task['combination']
        n_clusters = task['n_clusters']
        
        print(f"\n{'='*60}")
        print(f"▶️  Processing feature combination: {combination} ({n_clusters} clusters)")
        print(f"{'='*60}")

        # Load clustering results file for this combination
        cluster_file_path = os.path.join(CLUSTER_BASE_DIR, combination, 'kmeans', 'clustered_results.csv')
        try:
            cluster_df = pd.read_csv(cluster_file_path)
            print(f"  Successfully loaded clustering file: {cluster_file_path}")
        except FileNotFoundError:
            print(f"  ❌ Error: Clustering file not found {cluster_file_path}. Skipping this combination.")
            continue

        # Iterate through each frequency cutoff length
        for cutoff in FREQUENCY_CUTOFFS:
            print(f"\n  -- Processing frequency cutoff length: {cutoff} --")

            # 1. Define input and output paths for current task
            freq_file_path = os.path.join(FREQ_BASE_DIR, str(cutoff), 'materials_standardized_frequencies.csv')
            output_dir = os.path.join(CLUSTER_BASE_DIR, combination, 'avg_freq', str(cutoff))
            os.makedirs(output_dir, exist_ok=True)
            output_file_path = os.path.join(output_dir, 'average_frequencies.csv')

            # 2. Load frequency data for corresponding cutoff length
            try:
                freq_df = pd.read_csv(freq_file_path)
            except FileNotFoundError:
                print(f"    ❌ Error: Frequency file not found {freq_file_path}. Skipping this length.")
                continue
            
            # 3. Parse frequency strings to lists
            tqdm.pandas(desc=f"    Parsing frequencies (L={cutoff})")
            freq_df['standardized_frequency'] = freq_df['standardized_frequency'].progress_apply(safe_literal_eval)

            # 4. Merge clustering data and frequency data
            merged_df = pd.merge(cluster_df, freq_df, on='id', how='inner')
            if merged_df.empty:
                print("    ⚠️  Warning: Merged data is empty, cannot perform average calculation.")
                continue

            # 5. Group by cluster and calculate average frequencies
            avg_freq_results = []
            print(f"    Calculating average frequencies for {n_clusters} clusters...")
            for cluster_id in tqdm(range(n_clusters), desc="    Computing averages"):
                cluster_data = merged_df[merged_df['cluster'] == cluster_id]
                
                # Only calculate if cluster contains materials
                if not cluster_data.empty:
                    # Stack all material frequency lists in this cluster into a NumPy matrix
                    freq_matrix = np.array(cluster_data['standardized_frequency'].tolist())
                    
                    # Calculate average along columns (axis=0)
                    average_frequency = np.mean(freq_matrix, axis=0).tolist()
                    
                    avg_freq_results.append({
                        'cluster_id': cluster_id,
                        'average_frequency': average_frequency
                    })

            # 6. Save results
            if avg_freq_results:
                avg_df = pd.DataFrame(avg_freq_results)
                avg_df.to_csv(output_file_path, index=False)
                print(f"    ✅ Average frequency file saved to: {output_file_path}")
            else:
                print("    ⚠️  Warning: Could not calculate average frequencies for any cluster.")

    print("\n" + "="*60)
    print("✅ All tasks completed!")
    print("="*60)


if __name__ == "__main__":
    main()


=== Starting Average Phonon Frequency Calculation ===

▶️  Processing feature combination: 125 (50 clusters)
  Successfully loaded clustering file: ./clustering_results_umap/125/kmeans/clustered_results.csv

  -- Processing frequency cutoff length: 6 --


    Parsing frequencies (L=6): 100%|██████████| 11648/11648 [00:00<00:00, 58781.65it/s]


    Calculating average frequencies for 50 clusters...


    Computing averages: 100%|██████████| 50/50 [00:00<00:00, 1057.92it/s]


    ✅ Average frequency file saved to: ./clustering_results_umap/125/avg_freq/6/average_frequencies.csv

  -- Processing frequency cutoff length: 12 --


...

    ✅ Average frequency file saved to: ./clustering_results_umap/12345/avg_freq/54/average_frequencies.csv

  -- Processing frequency cutoff length: 60 --


    Parsing frequencies (L=60): 100%|██████████| 11648/11648 [00:01<00:00, 8778.06it/s]


    Calculating average frequencies for 45 clusters...


    Computing averages: 100%|██████████| 45/45 [00:00<00:00, 500.17it/s]

    ✅ Average frequency file saved to: ./clustering_results_umap/12345/avg_freq/60/average_frequencies.csv

✅ All tasks completed!


亲和传播聚类，以平均声子频率为参数，来进行聚类，目的是将第一步的簇，再一次分到亲和传播的类中，作为训练的数据集，亲和传播聚类的结果有几类，后续的训练就有几个数据集，即有几个模型。

In [4]:
import pandas as pd
import numpy as np
import os
import ast
import re
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AffinityPropagation
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics.pairwise import pairwise_distances
from umap import UMAP
from matplotlib.colors import ListedColormap
from tqdm import tqdm
import warnings

# Configuration
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# Global parameters
BASE_DIR = "./clustering_results_umap/"

# Define clustering tasks to process
TASKS = [
    {'combination': '125', 'n_clusters': 50},   
    {'combination': '235', 'n_clusters': 50},
    {'combination': '135', 'n_clusters': 55},
    {'combination': '12345', 'n_clusters': 45},
]

# Define frequency cutoff lengths to process
FREQUENCY_CUTOFFS = range(6, 61, 6)  # From 6 to 78, step 6


def safe_literal_eval(val):
    """Safely convert string representation of list or tuple to actual list."""
    if isinstance(val, str):
        try:
            evaluated_obj = ast.literal_eval(val)
            if isinstance(evaluated_obj, (list, tuple)):
                return list(evaluated_obj)
            return []
        except (ValueError, SyntaxError):
            return []
    return []


def export_origin_data(embedding_2d, meta_cluster_labels, valid_cluster_ids, material_counts, 
                       sample_silhouette_values, cluster_centers, output_dir, freq_length):
    """Export data files prepared for OriginLab"""
    
    # 1. Main scatter plot data
    scatter_data = pd.DataFrame({
        'x_coordinate': embedding_2d[:, 0],
        'y_coordinate': embedding_2d[:, 1],
        'original_cluster_id': valid_cluster_ids,
        'meta_cluster_id': meta_cluster_labels,
        'material_count': material_counts,
        'silhouette_score': sample_silhouette_values
    })
    scatter_data.to_csv(os.path.join(output_dir, f'origin_scatter_data_len{freq_length}.csv'), index=False)
    
    # 2. Cluster centers data
    centers_data = pd.DataFrame({
        'center_x': cluster_centers[:, 0],
        'center_y': cluster_centers[:, 1],
        'meta_cluster_id': range(len(cluster_centers))
    })
    centers_data.to_csv(os.path.join(output_dir, f'origin_centers_data_len{freq_length}.csv'), index=False)
    
    print(f"    ✅ Origin data files saved.")


def run_affinity_for_combination(combination, n_clusters):
    """Execute complete affinity propagation analysis pipeline for a single feature combination"""
    print(f"\n{'='*60}")
    print(f"▶️  Processing feature combination: {combination}")
    print(f"{'='*60}")
    
    clustering_summary = []

    # Iterate through each frequency cutoff length
    for cutoff in FREQUENCY_CUTOFFS:
        print(f"\n  -- Processing frequency cutoff length: {cutoff} --")

        # 1. Define paths
        avg_freq_file = os.path.join(BASE_DIR, combination, 'avg_freq', str(cutoff), 'average_frequencies.csv')
        stats_file = os.path.join(BASE_DIR, combination, 'kmeans', 'cluster_statistics.csv')
        output_dir = os.path.join(BASE_DIR, combination, 'affinity', str(cutoff))
        os.makedirs(output_dir, exist_ok=True)

        # 2. Load data
        try:
            avg_freq_df = pd.read_csv(avg_freq_file)
            stats_df = pd.read_csv(stats_file)
        except FileNotFoundError as e:
            print(f"    ❌ Error: Missing input file {e.filename}. Skipping this length.")
            continue
            
        # 3. Data preparation and merging
        avg_freq_df['average_frequency'] = avg_freq_df['average_frequency'].apply(safe_literal_eval)
        merged_data = pd.merge(avg_freq_df, stats_df, on='cluster_id')
        
        # Filter out clusters without frequencies or materials
        merged_data = merged_data[merged_data['average_frequency'].apply(len) > 0]
        merged_data = merged_data[merged_data['material_count'] > 0]

        if len(merged_data) < 3:
            print(f"    ⚠️  Warning: Insufficient valid data points ({len(merged_data)}). Skipping this length.")
            continue
            
        X = np.array(merged_data['average_frequency'].tolist())
        valid_cluster_ids = merged_data['cluster_id'].values
        material_counts = merged_data['material_count'].values

        # 4. UMAP dimensionality reduction
        reducer = UMAP(n_neighbors=min(15, len(X) - 1), min_dist=0.1, n_components=2, 
                      metric='euclidean', random_state=42)
        embedding_2d = reducer.fit_transform(X)

        # 5. Affinity propagation with parameter search
        distances = pairwise_distances(embedding_2d, metric='euclidean')
        similarities = -distances**2
        median_similarity = np.median(similarities)
        
        coef_list = [5.0, 4.0, 3.0, 2.5, 2.0, 1.5, 1.0, 0.8, 0.6, 0.5, 0.4]
        param_results = []

        for coef in coef_list:
            preference = coef * median_similarity
            ap = AffinityPropagation(damping=0.9, preference=preference, max_iter=400, random_state=42)
            try:
                cluster_labels = ap.fit_predict(embedding_2d)
                num_clusters = len(np.unique(cluster_labels))
                if 1 < num_clusters < len(X):
                    score = silhouette_score(embedding_2d, cluster_labels)
                    param_results.append({
                        'coefficient': coef, 'num_clusters': num_clusters,
                        'silhouette_score': score, 'labels': cluster_labels,
                        'centers': ap.cluster_centers_
                    })
            except Exception:
                continue
        
        if not param_results:
            print(f"    ❌ Error: Parameter search failed to find valid clustering results. Skipping this length.")
            continue
            
        # 6. Save best results
        best_result = max(param_results, key=lambda x: x['silhouette_score'])
        meta_cluster_labels = best_result['labels']
        cluster_centers = best_result['centers']
        num_meta_clusters = best_result['num_clusters']
        best_silhouette = best_result['silhouette_score']
        
        print(f"    Best result: {num_meta_clusters} meta-clusters, silhouette score: {best_silhouette:.4f}")

        # 7. Save detailed files
        sample_s_values = silhouette_samples(embedding_2d, meta_cluster_labels)
        
        # Save clustering results
        results_df = pd.DataFrame({
            'original_cluster_id': valid_cluster_ids,
            'meta_cluster': meta_cluster_labels,
            'material_count': material_counts,
            'silhouette_score': sample_s_values
        })
        results_df.to_csv(os.path.join(output_dir, 'clustering_results.csv'), index=False)
        
        # Create and save meta-cluster members file
        print(f"    Generating meta-cluster members file (cluster_members.csv)...")
        unique_meta_labels = sorted(np.unique(meta_cluster_labels))
        
        members_list = []
        for meta_id in unique_meta_labels:
            # Filter original cluster IDs belonging to current meta-cluster
            member_clusters = results_df[results_df['meta_cluster'] == meta_id]['original_cluster_id'].tolist()
            members_list.append({
                'meta_cluster_id': meta_id,
                'member_clusters': str(sorted(member_clusters)),  # Store as sorted list string
                'member_count': len(member_clusters)
            })
            
        members_df = pd.DataFrame(members_list)
        members_df.to_csv(os.path.join(output_dir, 'cluster_members.csv'), index=False)

        # Export Origin data
        export_origin_data(embedding_2d, meta_cluster_labels, valid_cluster_ids, material_counts,
                           sample_s_values, cluster_centers, output_dir, cutoff)
                           
        # Plot and save visualization
        plt.figure(figsize=(14, 12))
        colors = sns.color_palette('husl', n_colors=num_meta_clusters)
        cmap = ListedColormap(colors)
        
        scatter = plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=meta_cluster_labels, cmap=cmap,
                              s=np.sqrt(material_counts) * 10, alpha=0.8, edgecolors='black', linewidth=0.5)
        
        plt.scatter(cluster_centers[:, 0], cluster_centers[:, 1], s=150, c='black', marker='X', 
                   label='Cluster Centers')
        
        plt.title(f'Affinity Propagation on UMAP (Features: {combination}, Freq Length: {cutoff})', fontsize=16)
        plt.xlabel('UMAP Dimension 1', fontsize=12)
        plt.ylabel('UMAP Dimension 2', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.6)
        
        # Create legend
        legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                                     markerfacecolor=colors[i], markersize=10, 
                                     label=f'Meta-Cluster {i} (n={np.sum(meta_cluster_labels==i)})') 
                           for i in range(num_meta_clusters)]
        plt.legend(handles=legend_elements, title="Meta-Clusters", bbox_to_anchor=(1.02, 1), loc='upper left')
        
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        plt.savefig(os.path.join(output_dir, 'clustering_visualization.png'), dpi=300)
        plt.close()

        clustering_summary.append({
            'freq_length': cutoff,
            'num_meta_clusters': num_meta_clusters,
            'silhouette_score': best_silhouette
        })
    
    # 8. Generate summary report
    if clustering_summary:
        summary_df = pd.DataFrame(clustering_summary).sort_values(by='freq_length')
        summary_path = os.path.join(BASE_DIR, combination, 'affinity', f'affinity_summary_{combination}.csv')
        summary_df.to_csv(summary_path, index=False)
        print(f"\n  ✅ Summary report for feature combination {combination} saved to: {summary_path}")
        
        # Plot summary chart
        fig, ax1 = plt.subplots(figsize=(12, 7))
        ax2 = ax1.twinx()
        ax1.plot(summary_df['freq_length'], summary_df['silhouette_score'], 'o-', color='tab:blue', 
                label='Silhouette Score')
        ax2.plot(summary_df['freq_length'], summary_df['num_meta_clusters'], 's--', color='tab:red', 
                label='Number of Meta-Clusters')
        ax1.set_xlabel('Frequency Cutoff Length')
        ax1.set_ylabel('Silhouette Score', color='tab:blue')
        ax2.set_ylabel('Number of Meta-Clusters', color='tab:red')
        plt.title(f'Affinity Propagation Summary for Combination {combination}')
        fig.tight_layout()
        summary_plot_path = os.path.join(BASE_DIR, combination, 'affinity', f'affinity_summary_plot_{combination}.png')
        plt.savefig(summary_plot_path, dpi=300)
        plt.close()
        print(f"  ✅ Summary chart saved.")


def main():
    """Main execution pipeline"""
    print("=== Affinity Propagation Clustering Analysis Pipeline ===")
    
    for task in TASKS:
        run_affinity_for_combination(task['combination'], task['n_clusters'])
    
    print("\n" + "="*60)
    print("✅ All affinity propagation clustering tasks completed!")
    print("="*60)


if __name__ == "__main__":
    main()


=== Affinity Propagation Clustering Analysis Pipeline ===

▶️  Processing feature combination: 125

  -- Processing frequency cutoff length: 6 --
    Best result: 3 meta-clusters, silhouette score: 0.5683
    Generating meta-cluster members file (cluster_members.csv)...
    ✅ Origin data files saved.

  -- Processing frequency cutoff length: 12 --
    Best result: 2 meta-clusters, silhouette score: 0.6107
    Generating meta-cluster members file (cluster_members.csv)...
    ✅ Origin data files saved.

  -- Processing frequency cutoff length: 18 --
    Best result: 3 meta-clusters, silhouette score: 0.5411
    Generating meta-cluster members file (cluster_members.csv)...
    ✅ Origin data files saved.

  -- Processing frequency cutoff length: 24 --
    Best result: 2 meta-clusters, silhouette score: 0.6735
    Generating meta-cluster members file (cluster_members.csv)...
    ✅ Origin data files saved.

  -- Processing frequency cutoff length: 30 --
    Best result: 3 meta-clusters, silh

准备不同结构参数组合的test.csv文件用于最终测试 文件包含id、cluster以及frequency三列

In [ ]:
import pandas as pd
import os
import shutil
import json

# Configuration
DATASET_PATH = './dataset/dataset.csv'
BASE_OUTPUT_DIR = "./"  # 可修改

CLUSTERING_TASKS = [
    {'combination': '125', 'n_clusters': 50},   
    {'combination': '235', 'n_clusters': 50},
    {'combination': '135', 'n_clusters': 55},
    {'combination': '12345', 'n_clusters': 45},
]

# Process each combination
for task in CLUSTERING_TASKS:
    combination = task['combination']
    
    print(f"\n{'='*60}")
    print(f"Processing combination: {combination}")
    print(f"{'='*60}")
    
    # Source and destination paths
    source_file = os.path.join(BASE_OUTPUT_DIR, 'clustering_results_umap/', combination, 'kmeans', 'test_pre.csv')
    dest_dir = os.path.join(BASE_OUTPUT_DIR, 'test', combination)
    dest_file = os.path.join(dest_dir, 'test.csv')
    
    # Check if source file exists
    if not os.path.exists(source_file):
        print(f"  ⚠️  Warning: Source file not found: {source_file}")
        print(f"  Skipping combination {combination}")
        continue
    
    # Create destination directory
    os.makedirs(dest_dir, exist_ok=True)
    print(f"  Created directory: {dest_dir}")
    
    # Read test_pre.csv
    print(f"  Reading source file: {source_file}")
    test_df = pd.read_csv(source_file)
    print(f"  Loaded {len(test_df)} rows from test_pre.csv")
    
    # Add frequency column (initialize as empty string, will be filled from dataset)
    test_df['frequency'] = ''
    
    # Load dataset to get frequency data
    print(f"  Loading dataset: {DATASET_PATH}")
    try:
        dataset_df = pd.read_csv(DATASET_PATH, usecols=['id', 'frequency'])
        print(f"  Loaded {len(dataset_df)} rows from dataset")
        
        # Create a mapping dictionary for faster lookup
        frequency_map = dict(zip(dataset_df['id'], dataset_df['frequency']))
        
        # Fill frequency column by matching id
        print(f"  Matching frequencies by id...")
        matched_count = 0
        for idx, row in test_df.iterrows():
            material_id = row['id']
            if material_id in frequency_map:
                test_df.at[idx, 'frequency'] = frequency_map[material_id]
                matched_count += 1
            else:
                print(f"    ⚠️  Warning: ID {material_id} not found in dataset")
        
        print(f"  Matched {matched_count} out of {len(test_df)} rows")
        
    except FileNotFoundError:
        print(f"  ❌ Error: Dataset file not found: {DATASET_PATH}")
        print(f"  Frequency column will remain empty")
    except Exception as e:
        print(f"  ❌ Error loading dataset: {e}")
        print(f"  Frequency column will remain empty")
    
    # Save to destination
    test_df.to_csv(dest_file, index=False)
    print(f"  ✅ Saved test.csv to: {dest_file}")
    print(f"  File contains {len(test_df)} rows with columns: {test_df.columns.tolist()}")

print("\n" + "="*60)
print("✅ Testset for different combinations processed!")
print("="*60)
